# 🦟 West Nile Virus Surveillance — Colorado

**AEDES | Advanced Early Disease Prediction and Exploration Service**

This notebook tracks West Nile Virus (WNV) neuroinvasive disease in Colorado using:
- CDC NNDSS annual case reports
- NASA POWER daily temperature and precipitation data
- iNaturalist citizen-science mosquito observations

**Primary vector**: *Culex tarsalis* (western encephalitis mosquito)  
**Peak transmission**: July–September  
**Key risk factor**: Drought + heat waves concentrate birds and mosquitoes at shared water sources

In [ ]:
import json
import os
import datetime
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# Locate data directory relative to notebook
DATA_DIR = os.path.join(os.getcwd(), '..', 'data', 'surveillance')
TODAY = datetime.date.today().isoformat()

def load_json(filename):
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print(f'Analysis date: {TODAY}')
print(f'Data directory: {DATA_DIR}')
print(f'Data directory exists: {os.path.exists(DATA_DIR)}')

## 1. Annual Case Trends (2010–2024)

In [ ]:
raw = load_json('wnv_colorado.json')

if raw and raw.get('data'):
    df_wnv = pd.DataFrame(raw['data'])
    source_label = raw.get('source', 'CDC NNDSS')
    print(f'Source: {source_label}')
else:
    # Built-in sample data (CDC published historical values)
    print('Using built-in historical data (no data file found)')
    source_label = 'CDC NNDSS (built-in)'
    df_wnv = pd.DataFrame({
        'year':           [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
        'neuroinvasive':  [  51,   20,  130,   14,   43,   72,   14,    5,   15,   10,   10,    8,   16,    6,   12],
        'deaths':         [   3,    1,    9,    0,    2,    3,    1,    0,    0,    0,    1,    0,    0,    0,    0],
    })

print(f'Records: {len(df_wnv)}')
print(f'Total neuroinvasive cases (all years): {df_wnv["neuroinvasive"].sum()}')
print(f'Total deaths (all years): {df_wnv["deaths"].sum()}')
print(f'Peak year: {df_wnv.loc[df_wnv["neuroinvasive"].idxmax(), "year"]} ({df_wnv["neuroinvasive"].max()} cases)')
df_wnv.tail(5)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Colorado West Nile Virus Surveillance', fontsize=15, fontweight='bold', y=0.98)

# Neuroinvasive cases
colors = ['#e53e3e' if y == df_wnv.loc[df_wnv['neuroinvasive'].idxmax(), 'year'] else '#3182ce'
          for y in df_wnv['year']]
axes[0].bar(df_wnv['year'], df_wnv['neuroinvasive'], color=colors, alpha=0.85, zorder=3)
axes[0].set_ylabel('Neuroinvasive Cases', fontsize=11)
axes[0].set_title('Neuroinvasive Disease Cases by Year', fontsize=12)
axes[0].grid(axis='y', alpha=0.3, zorder=0)
axes[0].annotate('2012 outbreak\n(130 cases)', xy=(2012, 130), xytext=(2013.5, 120),
                 arrowprops=dict(arrowstyle='->', color='#e53e3e'),
                 fontsize=9, color='#e53e3e')

# Deaths
axes[1].bar(df_wnv['year'], df_wnv['deaths'], color='#742a2a', alpha=0.75, zorder=3)
axes[1].set_ylabel('Deaths', fontsize=11)
axes[1].set_title('WNV Deaths by Year', fontsize=12)
axes[1].grid(axis='y', alpha=0.3, zorder=0)
axes[1].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].set_xlabel('Year', fontsize=11)

plt.tight_layout()
plt.savefig('wnv_annual_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Source: {source_label}')

## 2. Recent Climate Conditions (90-Day Window)

In [ ]:
raw_climate = load_json('climate_colorado_90d.json')

if raw_climate and raw_climate.get('data') and len(raw_climate['data']) > 0:
    df_clim = pd.DataFrame(raw_climate['data'])
    df_clim['date'] = pd.to_datetime(df_clim['date'])
    df_clim = df_clim.dropna(subset=['temp_c'])
    climate_source = raw_climate.get('source', 'NASA POWER')
    print(f'Climate data: {len(df_clim)} days from {climate_source}')
    print(f'Date range: {df_clim["date"].min().date()} to {df_clim["date"].max().date()}')
    print(f'Mean temp (°C): {df_clim["temp_c"].mean():.1f}')
    days_above_18 = (df_clim['temp_c'] > 18).sum()
    print(f'Days above 18°C (WNV transmission threshold): {days_above_18}')
    have_climate = True
else:
    print('Climate API unavailable — showing transmission threshold reference only')
    have_climate = False

In [ ]:
if have_climate:
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    fig.suptitle('Colorado Climate Conditions — Last 90 Days (Denver)', fontsize=13, fontweight='bold')

    # Temperature with WNV threshold line
    axes[0].plot(df_clim['date'], df_clim['temp_c'], color='#dd6b20', linewidth=1.5, label='Daily temp (°C)')
    axes[0].axhline(18, color='#e53e3e', linestyle='--', linewidth=1.2, label='WNV transmission threshold (18°C)')
    axes[0].fill_between(df_clim['date'], df_clim['temp_c'], 18,
                         where=df_clim['temp_c'] > 18, alpha=0.15, color='#e53e3e', label='Above threshold')
    axes[0].set_ylabel('Temperature (°C)', fontsize=10)
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    # Precipitation
    if 'precip_mm' in df_clim.columns:
        axes[1].bar(df_clim['date'], df_clim['precip_mm'].fillna(0),
                    color='#3182ce', alpha=0.7, width=0.8, label='Precip (mm)')
        axes[1].set_ylabel('Precipitation (mm)', fontsize=10)
        axes[1].legend(fontsize=9)
        axes[1].grid(axis='y', alpha=0.3)
    axes[1].set_xlabel('Date', fontsize=10)

    plt.tight_layout()
    plt.savefig('wnv_climate_90d.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Source: {climate_source}')
else:
    print('Skipping climate chart (data unavailable)')

## 3. iNaturalist Mosquito Observations

In [ ]:
raw_inat = load_json('inaturalist_mosquitoes_colorado.json')

if raw_inat and raw_inat.get('data') and len(raw_inat['data']) > 0:
    df_inat = pd.DataFrame(raw_inat['data'])
    df_inat['observed_on'] = pd.to_datetime(df_inat['observed_on'], errors='coerce')
    df_inat = df_inat.dropna(subset=['observed_on'])

    print(f'iNaturalist mosquito observations (Colorado): {len(df_inat)}')
    print(f'Source: {raw_inat.get("source", "iNaturalist")}')
    if 'taxon' in df_inat.columns:
        print('\nTop species observed:')
        print(df_inat['taxon'].value_counts().head(8).to_string())

    # Monthly distribution
    df_inat['month'] = df_inat['observed_on'].dt.month
    monthly = df_inat.groupby('month').size().reindex(range(1, 13), fill_value=0)

    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(month_names, monthly.values, color='#2f855a', alpha=0.8)
    ax.set_title('iNaturalist Mosquito Observations by Month (Colorado)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Observations')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('inat_mosquitoes_monthly.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nFetched: {raw_inat.get("fetched", "unknown")}')
else:
    print('No iNaturalist data available (API unavailable or no observations returned)')

## 4. Early Warning Summary

In [ ]:
import calendar

current_month = datetime.date.today().month
current_month_name = calendar.month_name[current_month]

# Seasonal risk by month (based on Colorado WNV surveillance history)
monthly_risk = {
    1: ('Low',      '#48bb78'),
    2: ('Low',      '#48bb78'),
    3: ('Low',      '#48bb78'),
    4: ('Low',      '#48bb78'),
    5: ('Low',      '#48bb78'),
    6: ('Moderate', '#ed8936'),
    7: ('High',     '#e53e3e'),
    8: ('High',     '#e53e3e'),
    9: ('Moderate', '#ed8936'),
    10: ('Low',     '#48bb78'),
    11: ('Low',     '#48bb78'),
    12: ('Low',     '#48bb78'),
}

risk_level, risk_color = monthly_risk[current_month]

print('=' * 52)
print(f'  AEDES Early Warning Summary — {current_month_name}')
print('=' * 52)
print(f'  Seasonal WNV Risk Level : {risk_level}')
print(f'  Peak transmission months: July–September')
print(f'  Primary vector          : Culex tarsalis')
print(f'  Key amplifying hosts    : Corvids, house finches')
print()

if have_climate:
    recent_mean_temp = df_clim['temp_c'].tail(14).mean()
    print(f'  14-day mean temperature : {recent_mean_temp:.1f}°C')
    if recent_mean_temp > 18:
        print(f'  ⚠  Temperature above WNV transmission threshold (18°C)')
    else:
        print(f'  ✓  Temperature below WNV transmission threshold (18°C)')

print()
print(f'  Report: Colorado West Nile Virus: {df_wnv["neuroinvasive"].sum()} neuroinvasive cases')
print(f'          and {df_wnv["deaths"].sum()} deaths recorded 2010–2024')
print('=' * 52)
print(f'  Data sources: CDC NNDSS | NASA POWER | iNaturalist')
print(f'  Generated   : {TODAY}')